# ESIS-QUANT Layer 6 - Temporal Fusion Transformer (Kaggle GPU training)

**Self-contained**: embeds the *exact* from-scratch TFT source from the
`esis_quant` repo (grn.py / vsn.py / attention.py / model.py), so the saved
checkpoint loads back into the repo model via strict `load_state_dict`.
Trains on the real windows exported by `models.tft.features.export_features_npz`.

## How to run
1. **Settings -> Accelerator -> GPU T4**.
2. Upload `tft_features.npz` as a Dataset, *Add Input*, set `NPZ_PATH`.
3. **Run All**, then download `best_tft_model.pt` to
   `models/tft/checkpoints/best_tft_model.pt` in the repo.


In [ ]:
import os, math, types, numpy as np, torch
import torch.nn as nn

NPZ_PATH = "/kaggle/input/tft-features/tft_features.npz"
if not os.path.exists(NPZ_PATH):
    for c in ["tft_features.npz", "/kaggle/working/tft_features.npz"]:
        if os.path.exists(c): NPZ_PATH = c; break
print("NPZ:", NPZ_PATH)

EPOCHS=50; PATIENCE=10; BATCH=64; LR=1e-3; WD=1e-4; GRAD_CLIP=1.0
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "torch", torch.__version__)

# config shim: exactly the TFT_* constants the embedded repo code reads
config = types.SimpleNamespace(
    TFT_HIDDEN_SIZE=64, TFT_ATTENTION_HEADS=4, TFT_LSTM_LAYERS=1, TFT_DROPOUT=0.1,
    TFT_FORECAST_HORIZONS=[1,5,10,20], TFT_QUANTILES=[0.05,0.25,0.50,0.75,0.95],
)

In [ ]:
# ===== grn.py (verbatim) =====
class GatedLinearUnit(nn.Module):
    """GLU(x) = (W4 x + b4) * sigmoid(W5 x + b5)."""

    def __init__(self, input_size: int, output_size: int, dropout: float = 0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(input_size, output_size)
        self.gate = nn.Linear(input_size, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dropout(x)
        return self.fc(x) * torch.sigmoid(self.gate(x))


class GatedResidualNetwork(nn.Module):
    """GRN(a, c) = LayerNorm(skip(a) + GLU(W1 ELU(W2 a + W3 c + b2) + b1)).

    input_size : dim of the primary input a.
    hidden_size: inner ELU width.
    output_size: output dim (defaults to input_size). A linear skip
                 projection is added only when input_size != output_size.
    context_size: dim of the optional static context c (None = no context).
    """

    def __init__(self, input_size: int, hidden_size: int, output_size: int = None,
                 dropout: float = 0.1, context_size: int = None):
        super().__init__()
        output_size = output_size or input_size
        self.output_size = output_size

        self.fc2 = nn.Linear(input_size, hidden_size)          # W2 a
        self.context = nn.Linear(context_size, hidden_size, bias=False) if context_size else None  # W3 c
        self.elu = nn.ELU()
        self.fc1 = nn.Linear(hidden_size, hidden_size)          # W1
        self.glu = GatedLinearUnit(hidden_size, output_size, dropout)
        self.layer_norm = nn.LayerNorm(output_size)

        self.skip = nn.Linear(input_size, output_size) if input_size != output_size else None

    def forward(self, a: torch.Tensor, context: torch.Tensor = None) -> torch.Tensor:
        residual = a if self.skip is None else self.skip(a)
        eta2 = self.fc2(a)
        if self.context is not None and context is not None:
            # context is static (no time dim); broadcast over any middle dims of a
            while context.dim() < eta2.dim():
                context = context.unsqueeze(1)
            eta2 = eta2 + self.context(context)
        eta2 = self.elu(eta2)
        eta1 = self.fc1(eta2)
        return self.layer_norm(residual + self.glu(eta1))

In [ ]:
# ===== vsn.py (verbatim) =====
class VariableSelectionNetwork(nn.Module):
    """Select + combine `n_vars` scalar-per-variable inputs into one
    hidden_size vector, with interpretable per-variable softmax weights.

    Input to forward: x of shape (..., n_vars) where each of the last-dim
    entries is one variable's value at that (batch[, time]) position. Each
    variable is embedded independently to hidden_size before selection.
    """

    def __init__(self, n_vars: int, hidden_size: int, dropout: float = 0.1,
                 context_size: int = None):
        super().__init__()
        self.n_vars = n_vars
        self.hidden_size = hidden_size

        # per-variable embedding of a scalar -> hidden_size, then a GRN
        self.var_grns = nn.ModuleList([
            GatedResidualNetwork(1, hidden_size, hidden_size, dropout) for _ in range(n_vars)
        ])
        # weight network over the flattened raw inputs (+ optional static context)
        self.flattened_grn = GatedResidualNetwork(
            n_vars, hidden_size, n_vars, dropout, context_size=context_size
        )
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x: torch.Tensor, context: torch.Tensor = None):
        # x: (..., n_vars). Build selection weights from the raw flattened vars.
        weights = self.softmax(self.flattened_grn(x, context))  # (..., n_vars)

        # embed each variable independently
        var_outputs = []
        for v in range(self.n_vars):
            xi_v = self.var_grns[v](x[..., v:v + 1])  # (..., hidden_size)
            var_outputs.append(xi_v)
        stacked = torch.stack(var_outputs, dim=-2)  # (..., n_vars, hidden_size)

        # weighted sum over variables
        selected = (weights.unsqueeze(-1) * stacked).sum(dim=-2)  # (..., hidden_size)
        return selected, weights

In [ ]:
# ===== attention.py (verbatim) =====
class InterpretableMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.q_layers = nn.ModuleList([nn.Linear(d_model, self.d_k) for _ in range(n_heads)])
        self.k_layers = nn.ModuleList([nn.Linear(d_model, self.d_k) for _ in range(n_heads)])
        self.v_layer = nn.Linear(d_model, self.d_k)  # SHARED across heads (interpretability)
        self.w_o = nn.Linear(self.d_k, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor = None):
        """q,k,v: (B, T, d_model). mask: (T, T) or (B, T, T) with True where
        attention is DISALLOWED. Returns (output (B,T,d_model),
        attention (B,T,T) averaged over heads).
        """
        v_shared = self.v_layer(v)  # (B, T, d_k)
        head_outputs = []
        attns = []
        for h in range(self.n_heads):
            qh = self.q_layers[h](q)  # (B,T,d_k)
            kh = self.k_layers[h](k)
            scores = qh @ kh.transpose(-2, -1) / math.sqrt(self.d_k)  # (B,T,T)
            if mask is not None:
                scores = scores.masked_fill(mask, float("-inf"))
            attn = torch.softmax(scores, dim=-1)
            attn = torch.nan_to_num(attn, nan=0.0)
            attn = self.dropout(attn)
            head_outputs.append(attn @ v_shared)  # (B,T,d_k)
            attns.append(attn)

        # average heads (shared V makes this average meaningful)
        head_mean = torch.stack(head_outputs, dim=0).mean(dim=0)  # (B,T,d_k)
        attn_mean = torch.stack(attns, dim=0).mean(dim=0)          # (B,T,T)
        output = self.w_o(head_mean)
        return output, attn_mean


def causal_mask(seq_len: int, device=None) -> torch.Tensor:
    """(seq_len, seq_len) boolean mask, True in the strictly-upper triangle,
    so position t cannot attend to any position > t (no look-ahead).
    """
    return torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1)

In [ ]:
# ===== model.py (verbatim) =====
class _AddNorm(nn.Module):
    """Gated skip connection: LayerNorm(residual + GLU(x))."""

    def __init__(self, size: int, dropout: float):
        super().__init__()
        self.glu = GatedLinearUnit(size, size, dropout)
        self.norm = nn.LayerNorm(size)

    def forward(self, x: torch.Tensor, residual: torch.Tensor) -> torch.Tensor:
        return self.norm(residual + self.glu(x))


class TemporalFusionTransformer(nn.Module):
    def __init__(self, n_temporal: int, n_static: int,
                 hidden_size: int = None, n_heads: int = None, lstm_layers: int = None,
                 dropout: float = None, horizons=None, quantiles=None):
        super().__init__()
        H = hidden_size or config.TFT_HIDDEN_SIZE
        n_heads = n_heads or config.TFT_ATTENTION_HEADS
        lstm_layers = lstm_layers or config.TFT_LSTM_LAYERS
        dropout = config.TFT_DROPOUT if dropout is None else dropout
        self.horizons = list(horizons or config.TFT_FORECAST_HORIZONS)
        self.quantiles = list(quantiles or config.TFT_QUANTILES)
        self.hidden_size = H
        self.n_temporal = n_temporal
        self.n_static = n_static
        self.lstm_layers = lstm_layers

        # --- static path ---
        self.static_vsn = VariableSelectionNetwork(n_static, H, dropout) if n_static > 0 else None
        static_ctx_in = H if n_static > 0 else 0
        # 4 static context encoders (selection / enrichment / lstm h / lstm c)
        make_ctx = lambda: GatedResidualNetwork(H, H, H, dropout) if n_static > 0 else None
        self.ctx_selection = make_ctx()
        self.ctx_enrichment = make_ctx()
        self.ctx_h = make_ctx()
        self.ctx_c = make_ctx()

        # --- temporal path ---
        self.temporal_vsn = VariableSelectionNetwork(
            n_temporal, H, dropout, context_size=static_ctx_in or None
        )
        self.lstm = nn.LSTM(H, H, num_layers=lstm_layers, batch_first=True, dropout=0.0)
        self.post_lstm = _AddNorm(H, dropout)
        self.enrichment = GatedResidualNetwork(H, H, H, dropout, context_size=static_ctx_in or None)
        self.attention = InterpretableMultiHeadAttention(H, n_heads, dropout)
        self.post_attn = _AddNorm(H, dropout)
        self.position_wise = GatedResidualNetwork(H, H, H, dropout)
        self.post_ff = _AddNorm(H, dropout)

        # --- quantile heads: one per horizon, each outputs len(quantiles) ---
        self.quantile_heads = nn.ModuleList([
            nn.Linear(H, len(self.quantiles)) for _ in self.horizons
        ])

    def forward(self, past: torch.Tensor, static: torch.Tensor = None):
        """past: (B, T, n_temporal). static: (B, n_static) or None.
        Returns (quantile_preds (B, n_horizons, n_quantiles), interpretability dict).
        """
        B, T, _ = past.shape
        device = past.device

        # static contexts
        c_s = c_e = None
        static_weights = None
        if self.static_vsn is not None and static is not None:
            static_emb, static_weights = self.static_vsn(static)  # (B,H), (B,n_static)
            c_s = self.ctx_selection(static_emb)
            c_e = self.ctx_enrichment(static_emb)
            c_h = self.ctx_h(static_emb).unsqueeze(0).repeat(self.lstm_layers, 1, 1)  # (L,B,H)
            c_c = self.ctx_c(static_emb).unsqueeze(0).repeat(self.lstm_layers, 1, 1)
        else:
            c_h = torch.zeros(self.lstm_layers, B, self.hidden_size, device=device)
            c_c = torch.zeros(self.lstm_layers, B, self.hidden_size, device=device)

        # temporal variable selection (context = static selection context)
        selected, temporal_weights = self.temporal_vsn(past, c_s)  # (B,T,H), (B,T,n_temporal)

        # LSTM encoder, initialized from static contexts
        lstm_out, _ = self.lstm(selected, (c_h, c_c))
        temporal = self.post_lstm(lstm_out, selected)  # gated skip over the VSN output

        # static enrichment (per timestep)
        enriched = self.enrichment(temporal, c_e)

        # interpretable self-attention over time (causal: no look-ahead)
        mask = causal_mask(T, device=device)
        attn_out, attention = self.attention(enriched, enriched, enriched, mask=mask)
        attended = self.post_attn(attn_out, enriched)

        # position-wise feed-forward + final gated skip
        ff = self.position_wise(attended)
        out = self.post_ff(ff, temporal)  # skip back to the pre-enrichment temporal

        # forecast from the LAST observed timestep
        last = out[:, -1, :]  # (B,H)
        preds = torch.stack([head(last) for head in self.quantile_heads], dim=1)  # (B, n_horizons, n_quantiles)

        interpretability = {
            "temporal_weights": temporal_weights,   # (B,T,n_temporal)
            "static_weights": static_weights,       # (B,n_static) or None
            "attention": attention,                 # (B,T,T)
        }
        return preds, interpretability

In [ ]:
def pinball_loss(preds, targets, quantiles):
    q = torch.tensor(quantiles, dtype=preds.dtype, device=preds.device).view(1,1,-1)
    e = targets.unsqueeze(-1) - preds
    return torch.maximum(q*e, (q-1)*e).mean()

z = np.load(NPZ_PATH, allow_pickle=True)
temporal_cols=[str(x) for x in z["temporal_cols"]]; static_cols=[str(x) for x in z["static_cols"]]
horizons=[int(x) for x in z["horizons"]]; quantiles=[float(x) for x in z["quantiles"]]
def sp(n): return z[f"{n}_past"], z[f"{n}_static"], z[f"{n}_targets"]
Xtr,Str,Ytr=sp("train"); Xva,Sva,Yva=sp("val"); Xte,Ste,Yte=sp("test")
te_dates=z["test_dates"]; te_reg=z["test_regimes"]; te_tick=z["test_tickers"]
print("train",Xtr.shape,"val",Xva.shape,"test",Xte.shape)

from torch.utils.data import TensorDataset, DataLoader
def dl(X,S,Y,sh):
    return DataLoader(TensorDataset(torch.tensor(X,dtype=torch.float32),torch.tensor(S,dtype=torch.float32),
                                    torch.tensor(Y,dtype=torch.float32)), batch_size=BATCH, shuffle=sh)
tr=dl(Xtr,Str,Ytr,True); va=dl(Xva,Sva,Yva,False)

In [ ]:
# ============ PRE-TRAIN MANIFEST (inspect before trusting anything) ============
print("features used (n_temporal=%d):" % len(temporal_cols), temporal_cols)
print("static (%d):" % len(static_cols), static_cols)
for nm,X in [("train",Xtr),("val",Xva),("test",Xte)]:
    print(f"{nm}: {X.shape[0]} windows")
print("\nper-feature stats on TRAIN windows (standardized):")
zt=Xtr.reshape(-1,Xtr.shape[2])
for i,c in enumerate(temporal_cols):
    col=zt[:,i]; print(f"  {c:22s} std={col.std():.3f} min={col.min():+.2f} max={col.max():+.2f} frac_zero={(col==0).mean():.3f}")

# base rate (positive fraction) per horizon on TEST -- so a reviewer never has to reconstruct it
print("\ntest positive-rate base rate per horizon:", {h: round(float((Yte[:,i]>0).mean()),3) for i,h in enumerate(horizons)})

# ---- BASELINES (Fix 4): majority-class direction + unconditional-quantile pinball ----
train_majority_sign={h: np.sign(Ytr[:,i].mean()) for i,h in enumerate(horizons)}
baseline_dir_acc={h: round(float((np.sign(Yte[:,i])==train_majority_sign[h]).mean()),3) for i,h in enumerate(horizons)}
naive_q=np.array([[np.quantile(Ytr[:,i],q) for q in quantiles] for i in range(len(horizons))])  # (H,Q) from TRAIN only
def naive_pinball(Y):
    l=[np.maximum(q*(Y[:,i]-naive_q[i,j]),(q-1)*(Y[:,i]-naive_q[i,j])) for j,q in enumerate(quantiles) for i in range(len(horizons))]
    return float(np.concatenate(l).mean())
print("\nBASELINE majority-class direction acc:", baseline_dir_acc)
print("BASELINE naive-quantile pinball (val):", round(naive_pinball(Yva),5), " (test):", round(naive_pinball(Yte),5))
print(">>> the model must BEAT these to be worth anything <<<")

In [ ]:
model = TemporalFusionTransformer(len(temporal_cols), len(static_cols),
                                  horizons=horizons, quantiles=quantiles).to(DEVICE)
print("params:", sum(p.numel() for p in model.parameters()))
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=LR,epochs=EPOCHS,steps_per_epoch=max(1,len(tr)))
best=float("inf"); best_state=None; wait=0; history=[]
for ep in range(EPOCHS):
    model.train(); tl=0.0
    for X,S,Y in tr:
        X,S,Y=X.to(DEVICE),S.to(DEVICE),Y.to(DEVICE)
        loss=pinball_loss(model(X,S)[0],Y,quantiles)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); opt.step(); sched.step()
        tl+=loss.item()*len(X)
    tl/=len(Xtr); model.eval(); vl=0.0
    with torch.no_grad():
        for X,S,Y in va:
            X,S,Y=X.to(DEVICE),S.to(DEVICE),Y.to(DEVICE)
            vl+=pinball_loss(model(X,S)[0],Y,quantiles).item()*len(X)
    vl/=len(Xva); history.append({"epoch":ep,"train_loss":tl,"val_loss":vl})
    print(f"epoch {ep+1:2d}/{EPOCHS} train={tl:.5f} val={vl:.5f}")
    if vl<best-1e-6: best,best_state,wait=vl,{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},0
    else:
        wait+=1
        if wait>=PATIENCE: print("early stop"); break
if best_state: model.load_state_dict(best_state)
print("best val:", best)

In [ ]:
# ============ EVALUATION: model vs baselines, per horizon / regime / ticker ============
model.eval()
with torch.no_grad():
    pr=[]
    for i in range(0,len(Xte),512):
        pr.append(model(torch.tensor(Xte[i:i+512],dtype=torch.float32).to(DEVICE),
                        torch.tensor(Ste[i:i+512],dtype=torch.float32).to(DEVICE))[0].cpu().numpy())
    preds=np.concatenate(pr)
med=quantiles.index(0.5)

def dir_acc(p,Y): return {h:round(float((np.sign(p[:,i,med])==np.sign(Y[:,i])).mean()),3) for i,h in enumerate(horizons)}
def cover_per_h(p,Y):  # per-horizon (NOT pooled) coverage
    return {h:{round(q,2):round(float((Y[:,i]<=p[:,i,j]).mean()),3) for j,q in enumerate(quantiles)} for i,h in enumerate(horizons)}
def pinball(p,Y):
    q=np.array(quantiles).reshape(1,1,-1); e=Y[:,:,None]-p
    return float(np.maximum(q*e,(q-1)*e).mean())

model_da=dir_acc(preds,Yte); model_pin=pinball(preds,Yte)
print("=== MODEL vs BASELINE (test) ===")
print("directional acc  model:", model_da)
print("directional acc  base :", baseline_dir_acc)
print("  -> beats baseline?  ", {h:("YES" if model_da[h]>baseline_dir_acc[h] else "no") for h in horizons})
print("pinball  model:", round(model_pin,5), " naive:", round(naive_pinball(Yte),5),
      "->", ("model better" if model_pin<naive_pinball(Yte) else "NAIVE BETTER (model adds nothing)"))
print("\nper-horizon quantile coverage (nominal .05/.25/.5/.75/.95):")
for h,cov in cover_per_h(preds,Yte).items(): print(f"  {h:2d}d:", cov)

print("\n=== BY REGIME (directional acc, per horizon) ===")
for reg in ["BULL","BEAR","HIGH_VOL","RECOVERY","UNKNOWN"]:
    m=te_reg==reg
    if m.sum()>50: print(f"  {reg:9s} n={int(m.sum()):5d}:", dir_acc(preds[m],Yte[m]))

print("\n=== BY TICKER (5d directional acc, sorted) ===")
h5=horizons.index(5) if 5 in horizons else 0
rows=[]
for tk in np.unique(te_tick):
    m=te_tick==tk
    if m.sum()>30: rows.append((tk,round(float((np.sign(preds[m,h5,med])==np.sign(Yte[m,h5])).mean()),3),int(m.sum())))
for tk,acc,n in sorted(rows,key=lambda r:-r[1]): print(f"  {tk:6s} {acc:.3f}  (n={n})")

ckpt={"model_state_dict":model.state_dict(),"n_temporal":len(temporal_cols),"n_static":len(static_cols),
      "temporal_cols":temporal_cols,"static_cols":static_cols,"horizons":horizons,"quantiles":quantiles,"history":history}
torch.save(ckpt,"/kaggle/working/best_tft_model.pt")
print("\nsaved -> place at models/tft/checkpoints/best_tft_model.pt")